# 实验 9 — dlt 增量游标（`dlt.sources.incremental`）

**目标**：理解生产环境 dlt 管道最重要的模式 —— cursor-based incremental load。

现有的 `ecb_rates.py` 每次都拉全量（`/2024-01-01..today`），靠 `merge` 来去重。生产上这么做有问题：

- 上游 API 有限流，每次跑都重新拉全量会被 ban
- 每次传输/存储都花钱
- merge 也得全表扫描去重

**正确姿势**：dlt 记住上一次看到的 `max(date)`（high-water mark），下次只拉增量。State 持久化在 `_dlt_pipeline_state` 表里。

文件：[dlt_pipelines/ecb_rates_incremental.py](../dlt_pipelines/ecb_rates_incremental.py)。它用 `dataset_name='raw_ecb_inc'`，不会污染原有 `raw_ecb`。

In [ ]:
import subprocess, duckdb
DB = '../data/sandbox.duckdb'

def run():
    r = subprocess.run(['uv','run','python','dlt_pipelines/ecb_rates_incremental.py'],
                       capture_output=True, text=True, cwd='..')
    return r.stdout + r.stderr

print('=== 首次运行（拉全量）===')
print(run()[-800:])

In [ ]:
con = duckdb.connect(DB)
print('rows in raw_ecb_inc.daily_rates:',
      con.execute('select count(*) from raw_ecb_inc.daily_rates').fetchone()[0])
print('max(date):',
      con.execute('select max(date) from raw_ecb_inc.daily_rates').fetchone()[0])
print()
print('=== state 存在哪里？===')
con.execute("""
    select pipeline_name, created_at
    from raw_ecb_inc._dlt_pipeline_state
    order by created_at desc limit 3
""").df()

In [ ]:
# 第二次运行：注意输出里 rows 几乎为 0（只追最近的）
print('=== 第二次运行（增量）===')
print(run()[-800:])
print()
print('rows now:', con.execute('select count(*) from raw_ecb_inc.daily_rates').fetchone()[0])

In [ ]:
# 看看持久化的 cursor 值
import json
row = con.execute("""
    select state from raw_ecb_inc._dlt_pipeline_state
    order by created_at desc limit 1
""").fetchone()
if row:
    state = json.loads(row[0]) if isinstance(row[0], str) else row[0]
    print(json.dumps(state, indent=2, default=str)[:2000])

## 生产环境踩坑提示

1. **`initial_value` 只在首次生效**：state 已存在时 `initial_value` 被忽略。改了它但游标已推进，不会重拉历史。要回灌：`uv run dlt pipeline ecb_rates_incremental drop --resources daily_rates` 或 `pipeline.run(..., refresh='drop_resources')`。
2. **游标列选哪个**：优先选 `updated_at` / `event_time` 这种上游改动会随之更新的列。用 `created_at` 会漏掉对老记录的修改。
3. **边界与重叠**：`last_value` 默认严格大于，可能漏掉同时间戳的多条。`end_value` 可限定上界做回灌。
4. **类型一致性**：`initial_value` 与 `last_value` 会被传到下游（SQL / REST 参数），类型不对会静默失败（如该传 epoch ms 却传了字符串）。
5. **`merge` + incremental 是黄金组合**：增量减量 + merge 处理 late-arriving / corrected rows。

## 思考题

- 把 `initial_value` 改成 `2024-06-01`、drop 掉 state 后重跑，`_dlt_pipeline_state` 里看到什么？
- 上游 API 返回了 3 天前的修正值（`date < last_value`），增量拉取会丢掉它。怎么补救？（提示：lookback window 或定期全量重跑）
- 试试 `dlt.sources.incremental("date", initial_value=..., end_value=...)` 做指定时间区间的回填批次。